### First step to PySpark
Displaying a table

In [0]:
# Load and display the big_mart_sales table
big_mart_sales = spark.table("workspace.default.big_mart_sales")
display(big_mart_sales)

### - 2
Reading a CSV file and normalizes the data in columns.  
Remove the item identifier duplicates  
Writing to Parquet file and into a Delta Table (Only Delta Table is enough, the Parquet file is for test) 

In [0]:
from pyspark.sql import functions as F

df = spark.read.csv(
    "/Volumes/workspace/default/test/BigMart Sales.csv", header=True, inferSchema=True
)

df = df.withColumn(
    "Item_Fat_Content",
    F.when(F.col("Item_Fat_Content").isin("LF", "low fat"), F.lit("Low Fat"))
     .when(F.col("Item_Fat_Content").isin("reg", "regular"), F.lit("Regular"))
     .otherwise(F.col("Item_Fat_Content")),
)

df = df.dropDuplicates(['Item_Identifier'])

df.write.parquet("/Volumes/workspace/default/test/BigMart Sales.parquet", mode="overwrite")

df.write.format("delta").save("/Volumes/workspace/default/test/BigMart Sales.delta", mode="overwrite")
tables = spark.read.format("delta").load(
    "/Volumes/workspace/default/test/BigMart Sales.delta"
)

tables.createOrReplaceTempView("tables")

display(tables)

### - 3
Selecting the most selled outlet size with the number of outlet selled with total weight, average weight and average price.   

In [0]:
from pyspark.sql import functions as F

df = df.select("Outlet_Size", "Item_Outlet_Sales", "Item_Weight").groupBy("Outlet_Size").agg(
    F.round(F.sum("Item_Outlet_Sales"), 2).alias("Total_Sales"), 
    F.count("Item_Outlet_Sales").alias("Count"),
    F.round(F.sum("Item_Weight"), 2).alias("Total Weight"),
    F.round(F.sum("Item_Weight") / F.count("Item_Outlet_Sales"), 2).alias("Average Weight"),
    F.avg("Item_Outlet_Sales").alias("Average Outlet Price"),
)

df = df.orderBy("Total_Sales", ascending=False)

display(df)

### - 4
Testing windows functions

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

w = Window.partitionBy("Outlet_Size").orderBy(F.col("Total_Sales").desc())

df.display()

### - 5
Getting the files total size and comparing the size difference.  
We can see that by removing duplicates and serializing to Parquet makes the file size more than 16 times smaller than the CSV one.  
The Delta Table one has more size but it has the ACID properties that makes it safe, reliable.

In [0]:
def get_dir_size(path):
  total_size = 0
  try:
    for f in dbutils.fs.ls(path):
      if f.isDir():
        total_size += get_dir_size(f.path)
      else:
        total_size += f.size
  except Exception as e:
    print(f"Error: {e}")
  return total_size

base_path = "/Volumes/workspace/default/test"
files = dbutils.fs.ls(base_path)

for f in files:
  if f.isDir():
    size = get_dir_size(f.path)
  else:
    size = f.size
  print(f"{f.name}: {size} bytes ({size / (1024 * 1024):.2f} MB)")